In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import dynamiqs as dq

In [ ]:
# --- constants ---
ħ = 1
e = 1
phi_quantum = 1 / 2

w_a = 25.338776456203686
w_b = 2*w_a
kappa_b = np.pi / 10.4*2
E_J = 37*2*np.pi *1.5
phi_a = 0.11
phi_b = 0.204
epsilon_p = 1.1
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b 
alpha_sq = 1.5
epsilon_d = 2*alpha_sq*g
kappa_2 = 4*g**2/kappa_b
kappa_1 = 0.005*kappa_2

In [ ]:
n_a = 15
n_b = 2
N = n_a * n_b

# Operators
a = dq.destroy(n_a)
b = dq.destroy(n_b)
Ia = dq.eye(n_a)
Ib = dq.eye(n_b)

a_tot = dq.tensor(a, Ib)
b_tot = dq.tensor(Ia, b)

# Correct nonlinear term
term = dq.tensor(a @ a, dq.dag(b))
non_linear_op = term + dq.dag(term)
drive_op = b_tot + b_tot.dag()

La = np.sqrt(kappa_1) * a_tot
Lb = np.sqrt(kappa_b) * b_tot

# Convert to sparse CSC
La = sp.csc_matrix(La.to_numpy().astype(np.complex128))
Lb = sp.csc_matrix(Lb.to_numpy().astype(np.complex128))
I = sp.identity(N, dtype=np.complex128, format="csc")

def dissipator(L):
    LdL = L.getH() @ L
    return (
        sp.kron(L.conjugate(), L, format="csc")
        - 0.5 * sp.kron(I, LdL, format="csc")
        - 0.5 * sp.kron(LdL.T, I, format="csc")
    )
H = g * non_linear_op + (epsilon_d / 2) * drive_op
H = sp.csc_matrix(H.to_numpy().astype(np.complex128))
Lio = (
    -1j * (sp.kron(I, H, format="csc") - sp.kron(H.T, I, format="csc"))
    + dissipator(La)
    + dissipator(Lb)
    )

In [ ]:
vals, vecs = spla.eigs(Lio, k=3, sigma=0, which="LM", tol=1e-12)

In [ ]:
imag_tol = 1e-8

real_candidates = np.where(np.abs(vals.imag) < imag_tol)[0]
real_order = real_candidates[np.argsort(vals[real_candidates].real)[::-1]]

idx_zero = real_order[0]
idx_1 = real_order[1]

val_zero = vals[idx_zero]
val_1 = vals[idx_1]

vec_zero = vecs[:, idx_zero]
vec_1 = vecs[:, idx_1]

print((val_zero, val_1))

In [ ]:
rho_zero_a = dq.ptrace(dq.asqarray(vec_zero.reshape((n_a*n_b, n_a*n_b)), (n_a, n_b)),0)
ax_0 = dq.plot.wigner(rho_zero_a)
rho_1_a = dq.ptrace(dq.asqarray(vec_1.reshape((n_a*n_b, n_a*n_b)), (n_a, n_b)),0)
ax_1 = dq.plot.wigner(rho_1_a)

In [ ]:
import json

with open("/Users/tmalas/Desktop/Dev/Simulations_local/Code_python/Quantum_simulation_ATS/eigenvals_liouvillian/output_to_cluster/vec_1_re.json", "w") as f:
    json.dump(vec_1.real.tolist(), f)

with open("/Users/tmalas/Desktop/Dev/Simulations_local/Code_python/Quantum_simulation_ATS/eigenvals_liouvillian/output_to_cluster/vec_1_im.json", "w") as f:
    json.dump(vec_1.imag.tolist(), f)


In [ ]:
vals